## Environment

In [1]:
!pip install -q kagglehub pandas pillow tqdm


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data settings

In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataConfig:
    dataset_name: str = "maitam/vietnamese-traffic-signs"

    notebook_folder: str = "notebooks"
    image_folder: str = "traffic-signs-image"
    json_folder: str = "traffic-signs-json"

    processed_folder: str = "processed"
    checkpoint_folder: str = "checkpoints"
    result_folder: str = "results"
    figure_folder: str = "figures"

    train_file: str = "train.json"
    val_file: str = "val.json"
    test_file: str = "test.json"
    report_file: str = "report.json"
    meta_file: str = "meta.json"

    answer_limit: int = 10
    min_train_items: int = 2000
    min_test_items: int = 50
    min_train_images: int = 200
    min_questions_per_image: int = 3

    image_exts: tuple[str, ...] = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

    @property
    def notebook_dir(self) -> Path:
        return Path.cwd()

    @property
    def project_dir(self) -> Path:
        if self.notebook_dir.name == self.notebook_folder:
            return self.notebook_dir.parent

        return self.notebook_dir

    @property
    def image_dir(self) -> Path:
        return self.project_dir / self.image_folder

    @property
    def json_dir(self) -> Path:
        return self.project_dir / self.json_folder

    @property
    def processed_dir(self) -> Path:
        return self.project_dir / self.processed_folder

    @property
    def checkpoint_dir(self) -> Path:
        return self.project_dir / self.checkpoint_folder

    @property
    def result_dir(self) -> Path:
        return self.project_dir / self.result_folder

    @property
    def figure_dir(self) -> Path:
        return self.result_dir / self.figure_folder

## Path setup

In [3]:
import os
import shutil
from pathlib import Path

import kagglehub


class DataPaths:
    def __init__(self, config: DataConfig) -> None:
        self.config = config
        self.dataset_dir = None

    def make_outputs(self) -> None:
        folders = [
            self.config.image_dir,
            self.config.json_dir,
            self.config.processed_dir,
            self.config.checkpoint_dir,
            self.config.result_dir,
            self.config.figure_dir
        ]

        for folder in folders:
            folder.mkdir(parents=True, exist_ok=True)

    def has_images(self, folder: Path) -> bool:
        if not folder.exists() or not folder.is_dir():
            return False

        for path in folder.iterdir():
            if path.suffix.lower() in self.config.image_exts:
                return True

        return False

    def has_json(self) -> bool:
        files = [
            self.raw_train(),
            self.raw_val(),
            self.raw_test()
        ]

        return all(path.exists() for path in files)

    def dataset_root(self) -> Path:
        if self.has_images(self.config.image_dir):
            return self.config.project_dir

        if self.dataset_dir is None:
            self.dataset_dir = Path(
                kagglehub.dataset_download(self.config.dataset_name)
            )

        return self.dataset_dir

    def roots(self) -> list[Path]:
        root = self.dataset_root()
        roots = [
            root,
            root.parent
        ]
        clean = []

        for item in roots:
            if item.exists() and item not in clean:
                clean.append(item)

        return clean

    def image_count(self, folder: Path) -> int:
        count = 0

        for path in folder.iterdir():
            if path.suffix.lower() in self.config.image_exts:
                count += 1

        return count

    def image_source(self) -> Path:
        candidates = []

        for root in self.roots():
            for folder in root.rglob("*"):
                if folder.is_dir() and self.has_images(folder):
                    candidates.append(folder)

        if not candidates:
            raise FileNotFoundError("Cannot find image folder in dataset")

        candidates.sort(key=self.image_count, reverse=True)

        return candidates[0]

    def copy_images(self) -> None:
        if self.has_images(self.config.image_dir):
            return

        source = self.image_source()
        self.config.image_dir.mkdir(parents=True, exist_ok=True)

        for path in source.iterdir():
            if path.suffix.lower() not in self.config.image_exts:
                continue

            target = self.config.image_dir / path.name

            if not target.exists():
                shutil.copy2(path, target)

    def prepare(self) -> None:
        self.make_outputs()
        self.copy_images()

        if not self.has_json():
            raise FileNotFoundError("Missing train/val/test JSON files")

    def rel_path(self, path: Path) -> str:
        rel = os.path.relpath(path, start=self.config.notebook_dir)

        return Path(rel).as_posix()

    def raw_train(self) -> Path:
        return self.config.json_dir / self.config.train_file

    def raw_val(self) -> Path:
        return self.config.json_dir / self.config.val_file

    def raw_test(self) -> Path:
        return self.config.json_dir / self.config.test_file

    def out_train(self) -> Path:
        return self.config.processed_dir / self.config.train_file

    def out_val(self) -> Path:
        return self.config.processed_dir / self.config.val_file

    def out_test(self) -> Path:
        return self.config.processed_dir / self.config.test_file

    def report_path(self) -> Path:
        return self.config.processed_dir / self.config.report_file

    def meta_path(self) -> Path:
        return self.config.processed_dir / self.config.meta_file

## Read and format data

In [4]:
import json
from pathlib import Path


class JsonReader:
    @staticmethod
    def read(path: Path) -> object:
        if not path.exists():
            raise FileNotFoundError(f"Missing JSON file: {path}")

        with path.open("r", encoding="utf-8") as file:
            return json.load(file)

    @staticmethod
    def read_list(path: Path) -> list[dict[str, object]]:
        data = JsonReader.read(path)

        if not isinstance(data, list):
            raise ValueError(f"Expected list JSON: {path}")

        return data

    @staticmethod
    def write_new(data: object, path: Path) -> None:
        if path.exists():
            return

        path.parent.mkdir(parents=True, exist_ok=True)

        with path.open("w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)

In [5]:
import re
import unicodedata


class TextNorm:
    @staticmethod
    def normalize(text: str) -> str:
        text = unicodedata.normalize("NFC", str(text).lower().strip())
        text = re.sub(r"[^\w\sÀ-ỹ-]", "", text)
        text = text.replace("-", " ")
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    @staticmethod
    def tokens(text: str) -> list[str]:
        text = TextNorm.normalize(text)

        if not text:
            return []

        return text.split()

    @staticmethod
    def answer(value: object) -> str:
        if isinstance(value, list):
            if not value:
                return ""

            return str(value[0]).strip()

        return str(value).strip()

    @staticmethod
    def answer_count(value: object) -> int:
        if isinstance(value, list):
            return len(value)

        if value is None:
            return 0

        return 1

In [6]:
import re


class AnswerNorm:
    number_map = {
        "một": "1",
        "hai": "2",
        "ba": "3",
        "bốn": "4",
        "năm": "5",
        "sáu": "6",
        "bảy": "7",
        "tám": "8",
        "chín": "9",
        "mười": "10"
    }

    @staticmethod
    def clean(text: object) -> str:
        if isinstance(text, list):
            text = text[0] if text else ""

        text = str(text).strip()
        text = re.sub(r"\s+", " ", text)
        text = text.replace(" ,", ",")
        text = text.replace(" .", ".")

        return text

    @classmethod
    def finish(cls, text: str) -> str:
        text = cls.clean(text)

        if not text:
            return text

        if not text.endswith("."):
            text += "."

        return text[0].upper() + text[1:]

    @classmethod
    def number_words(cls, text: str) -> str:
        for word, number in cls.number_map.items():
            text = re.sub(
                rf"\b{word}\b",
                number,
                text,
                flags=re.IGNORECASE
            )

        return text

    @classmethod
    def yes_no(cls, answer: str) -> str:
        answer = cls.clean(answer).lower()

        if answer.startswith(("có", "đúng")):
            return "Có."

        if answer.startswith(("không", "sai")):
            return "Không."

        return cls.finish(answer)

    @classmethod
    def counting(cls, answer: str) -> str:
        answer = cls.number_words(cls.clean(answer))
        answer = re.sub(r"^chỉ có\s+", "Có ", answer, flags=re.IGNORECASE)
        answer = re.sub(r"^có\s+", "Có ", answer, flags=re.IGNORECASE)
        answer = re.sub(
            r"^Có\s+(\d+)\s+biển báo duy nhất",
            r"Có \1 biển báo",
            answer,
            flags=re.IGNORECASE
        )
        answer = re.sub(
            r"^Có\s+(\d+)\s+biển(?!\s*báo)",
            r"Có \1 biển báo",
            answer,
            flags=re.IGNORECASE
        )
        answer = re.sub(
            r"^(\d+)\s+(.+)$",
            r"Có \1 \2",
            answer,
            flags=re.IGNORECASE
        )
        answer = re.sub(
            r"^Có\s+(\d+)\s+biển(?!\s*báo)",
            r"Có \1 biển báo",
            answer,
            flags=re.IGNORECASE
        )

        return cls.finish(answer)

    @classmethod
    def identity(cls, answer: str) -> str:
        answer = cls.clean(answer)

        if not answer:
            return answer

        lower = answer.lower()

        if lower.startswith("biển báo"):
            answer = answer
        elif lower.startswith("biển "):
            answer = "Biển báo " + answer[5:]
        else:
            answer = "Biển báo " + answer[0].lower() + answer[1:]

        answer = re.sub(
            r"^Biển báo\s+biển\s+",
            "Biển báo ",
            answer,
            flags=re.IGNORECASE
        )

        return cls.finish(answer)

    @classmethod
    def attribute(cls, answer: str) -> str:
        answer = cls.clean(answer)
        replacements = {
            "có viền màu đỏ": "có viền đỏ",
            "viền màu đỏ": "viền đỏ",
            "màu sắc chủ đạo là": "màu",
            "màu chủ đạo là": "màu"
        }

        for old, new in replacements.items():
            answer = re.sub(old, new, answer, flags=re.IGNORECASE)

        answer = re.sub(
            r"^Màu\s+(.+)$",
            lambda match: match.group(1)[0].upper() + match.group(1)[1:],
            answer
        )
        answer = re.sub(
            r"\bvà\s+màu\s+",
            "và ",
            answer,
            flags=re.IGNORECASE
        )

        return cls.finish(answer)

    @classmethod
    def spatial(cls, answer: str) -> str:
        answer = cls.clean(answer)
        replacements = {
            "nằm ở phía bên phải đường": "Nằm bên phải đường",
            "nằm ở phía bên trái đường": "Nằm bên trái đường",
            "nằm ở phía bên phải": "Nằm bên phải",
            "nằm ở phía bên trái": "Nằm bên trái",
            "nằm ở bên phải": "Nằm bên phải",
            "nằm ở bên trái": "Nằm bên trái"
        }

        for old, new in replacements.items():
            answer = re.sub(old, new, answer, flags=re.IGNORECASE)

        return cls.finish(answer)

    @classmethod
    def normalize(cls, answer: object, q_type: object) -> str:
        answer = TextNorm.answer(answer)
        q_type = str(q_type).strip()

        if q_type == "Yes_No":
            return cls.yes_no(answer)

        if q_type == "Counting":
            return cls.counting(answer)

        if q_type == "Identity":
            return cls.identity(answer)

        if q_type == "Attribute":
            return cls.attribute(answer)

        if q_type == "Spatial":
            return cls.spatial(answer)

        return cls.finish(answer)

In [7]:
from pathlib import Path


class SampleFormat:
    @staticmethod
    def image_name(sample: dict[str, object]) -> str:
        value = sample.get("image_name")

        if value is not None:
            return str(value)

        return ""

    @staticmethod
    def one(
        sample: dict[str, object],
        index: int,
        split: str,
        paths: DataPaths
    ) -> dict[str, object]:
        image_name = SampleFormat.image_name(sample)
        image_path = paths.config.image_dir / image_name
        q_type = str(sample.get("type", "")).strip()
        answer = AnswerNorm.normalize(sample.get("answer"), q_type)

        return {
            "id": f"{split}_{index:06d}",
            "image": image_name,
            "image_name": image_name,
            "image_path": paths.rel_path(image_path),
            "question": str(sample.get("question", "")).strip(),
            "answer": answer,
            "type": q_type,
            "image_idx": sample.get("image_idx"),
            "group_id": sample.get("group_id"),
            "source_index": sample.get("index", index)
        }

    @staticmethod
    def all(
        samples: list[dict[str, object]],
        split: str,
        paths: DataPaths
    ) -> list[dict[str, object]]:
        items = []

        for index, sample in enumerate(samples):
            items.append(
                SampleFormat.one(
                    sample=sample,
                    index=index,
                    split=split,
                    paths=paths
                )
            )

        return items

## Validate data

In [8]:
from pathlib import Path


class DataCheck:
    @staticmethod
    def image_set(items: list[dict[str, object]]) -> set[str]:
        return {str(item.get("image", "")) for item in items}

    @staticmethod
    def type_count(items: list[dict[str, object]]) -> dict[str, int]:
        counts = {}

        for item in items:
            name = str(item.get("type", ""))
            counts[name] = counts.get(name, 0) + 1

        return counts

    @staticmethod
    def image_count(items: list[dict[str, object]]) -> dict[str, int]:
        counts = {}

        for item in items:
            name = str(item.get("image", ""))
            counts[name] = counts.get(name, 0) + 1

        return counts

    @staticmethod
    def issue_rows(
        items: list[dict[str, object]],
        raw: list[dict[str, object]],
        config: DataConfig
    ) -> dict[str, list[str]]:
        missing = []
        long_answer = []
        multi_answer = []
        missing_image = []

        for index, item in enumerate(items):
            row_id = str(item.get("id", ""))
            answer = str(item.get("answer", ""))
            image_path = config.notebook_dir / str(item.get("image_path", ""))

            if not item.get("image"):
                missing.append(row_id)

            if not item.get("question"):
                missing.append(row_id)

            if not item.get("answer"):
                missing.append(row_id)

            if not item.get("type"):
                missing.append(row_id)

            if len(TextNorm.tokens(answer)) > config.answer_limit:
                long_answer.append(row_id)

            if TextNorm.answer_count(raw[index].get("answer")) != 1:
                multi_answer.append(row_id)

            if not image_path.exists():
                missing_image.append(row_id)

        return {
            "missing_required_rows": missing,
            "long_answer_rows": long_answer,
            "multi_answer_rows": multi_answer,
            "missing_image_rows": missing_image
        }

    @staticmethod
    def split_summary(
        items: list[dict[str, object]],
        raw: list[dict[str, object]],
        config: DataConfig
    ) -> dict[str, object]:
        image_counts = DataCheck.image_count(items)
        values = list(image_counts.values())
        issues = DataCheck.issue_rows(items, raw, config)

        return {
            "items": len(items),
            "images": len(image_counts),
            "min_questions_per_image": min(values) if values else 0,
            "max_questions_per_image": max(values) if values else 0,
            "question_types": DataCheck.type_count(items),
            "issues": issues
        }

    @staticmethod
    def report(
        train: list[dict[str, object]],
        val: list[dict[str, object]],
        test: list[dict[str, object]],
        train_raw: list[dict[str, object]],
        val_raw: list[dict[str, object]],
        test_raw: list[dict[str, object]],
        config: DataConfig
    ) -> dict[str, object]:
        train_sum = DataCheck.split_summary(train, train_raw, config)
        val_sum = DataCheck.split_summary(val, val_raw, config)
        test_sum = DataCheck.split_summary(test, test_raw, config)

        train_images = DataCheck.image_set(train)
        val_images = DataCheck.image_set(val)
        test_images = DataCheck.image_set(test)

        issues = {
            "missing_required_rows": (
                train_sum["issues"]["missing_required_rows"]
                + val_sum["issues"]["missing_required_rows"]
                + test_sum["issues"]["missing_required_rows"]
            ),
            "long_answer_rows": (
                train_sum["issues"]["long_answer_rows"]
                + val_sum["issues"]["long_answer_rows"]
                + test_sum["issues"]["long_answer_rows"]
            ),
            "multi_answer_rows": (
                train_sum["issues"]["multi_answer_rows"]
                + val_sum["issues"]["multi_answer_rows"]
                + test_sum["issues"]["multi_answer_rows"]
            ),
            "missing_image_rows": (
                train_sum["issues"]["missing_image_rows"]
                + val_sum["issues"]["missing_image_rows"]
                + test_sum["issues"]["missing_image_rows"]
            ),
            "train_test_overlap_images": sorted(train_images & test_images),
            "train_val_overlap_images": sorted(train_images & val_images),
            "val_test_overlap_images": sorted(val_images & test_images)
        }

        rules = {
            "train_items_at_least_2000": (
                train_sum["items"] >= config.min_train_items
            ),
            "train_images_at_least_200": (
                train_sum["images"] >= config.min_train_images
            ),
            "test_items_at_least_50": (
                test_sum["items"] >= config.min_test_items
            ),
            "train_each_image_at_least_3_questions": (
                train_sum["min_questions_per_image"]
                >= config.min_questions_per_image
            ),
            "answer_length_at_most_10_words": (
                len(issues["long_answer_rows"]) == 0
            ),
            "one_answer_per_question": (
                len(issues["multi_answer_rows"]) == 0
            ),
            "no_train_test_image_overlap": (
                len(issues["train_test_overlap_images"]) == 0
            ),
            "all_image_files_exist": (
                len(issues["missing_image_rows"]) == 0
            )
        }

        return {
            "summary": {
                "train_items": train_sum["items"],
                "val_items": val_sum["items"],
                "test_items": test_sum["items"],
                "train_images": train_sum["images"],
                "val_images": val_sum["images"],
                "test_images": test_sum["images"],
                "total_items": (
                    train_sum["items"]
                    + val_sum["items"]
                    + test_sum["items"]
                ),
                "total_images": len(train_images | val_images | test_images)
            },
            "question_types": {
                "train": train_sum["question_types"],
                "val": val_sum["question_types"],
                "test": test_sum["question_types"]
            },
            "rules": rules,
            "issues": issues
        }

## Build processed data

In [9]:
import json
from pathlib import Path


class DataBuild:
    def __init__(self, config: DataConfig, paths: DataPaths) -> None:
        self.config = config
        self.paths = paths
        self.paths.prepare()

    def done(self) -> bool:
        files = [
            self.paths.out_train(),
            self.paths.out_val(),
            self.paths.out_test(),
            self.paths.report_path(),
            self.paths.meta_path()
        ]

        return all(path.exists() for path in files)

    def load_raw(self) -> tuple[
        list[dict[str, object]],
        list[dict[str, object]],
        list[dict[str, object]]
    ]:
        return (
            JsonReader.read_list(self.paths.raw_train()),
            JsonReader.read_list(self.paths.raw_val()),
            JsonReader.read_list(self.paths.raw_test())
        )

    def format_data(
        self,
        train_raw: list[dict[str, object]],
        val_raw: list[dict[str, object]],
        test_raw: list[dict[str, object]]
    ) -> tuple[
        list[dict[str, object]],
        list[dict[str, object]],
        list[dict[str, object]]
    ]:
        train = SampleFormat.all(train_raw, "train", self.paths)
        val = SampleFormat.all(val_raw, "val", self.paths)
        test = SampleFormat.all(test_raw, "test", self.paths)

        return train, val, test

    def meta(self) -> dict[str, str]:
        return {
            "image_dir": self.paths.rel_path(self.config.image_dir),
            "train_json": self.paths.rel_path(self.paths.out_train()),
            "val_json": self.paths.rel_path(self.paths.out_val()),
            "test_json": self.paths.rel_path(self.paths.out_test()),
            "checkpoint_dir": self.paths.rel_path(self.config.checkpoint_dir),
            "result_dir": self.paths.rel_path(self.config.result_dir),
            "figure_dir": self.paths.rel_path(self.config.figure_dir)
        }

    def save(
        self,
        train: list[dict[str, object]],
        val: list[dict[str, object]],
        test: list[dict[str, object]],
        report: dict[str, object]
    ) -> None:
        JsonReader.write_new(train, self.paths.out_train())
        JsonReader.write_new(val, self.paths.out_val())
        JsonReader.write_new(test, self.paths.out_test())
        JsonReader.write_new(report, self.paths.report_path())
        JsonReader.write_new(self.meta(), self.paths.meta_path())

    def run(self) -> dict[str, object]:
        if self.done():
            return JsonReader.read(self.paths.report_path())

        train_raw, val_raw, test_raw = self.load_raw()
        train, val, test = self.format_data(
            train_raw,
            val_raw,
            test_raw
        )
        report = DataCheck.report(
            train=train,
            val=val,
            test=test,
            train_raw=train_raw,
            val_raw=val_raw,
            test_raw=test_raw,
            config=self.config
        )
        self.save(train, val, test, report)

        return report

## Run

In [10]:
import pandas as pd


config = DataConfig()
paths = DataPaths(config)
build = DataBuild(config, paths)

report = build.run()
pd.DataFrame(JsonReader.read_list(paths.out_train())).head()

,id,image,image_name,image_path,question,answer,type,image_idx,group_id,source_index
0,train_000000,0675.jpg,0675.jpg,../traffic-signs-image/0675.jpg,Biển báo trên cùng là biển gì?,Biển báo cấm rẽ phải và quay đầu.,Identity,675,1,0
1,train_000001,0675.jpg,0675.jpg,../traffic-signs-image/0675.jpg,Biển báo cấm có hình dạng và màu sắc gì?,"Hình tròn, viền đỏ, nền trắng.",Attribute,675,1,1
2,train_000002,0675.jpg,0675.jpg,../traffic-signs-image/0675.jpg,Trong ảnh có biển báo cấm rẽ trái không?,Có.,Yes_No,675,1,2
3,train_000003,0675.jpg,0675.jpg,../traffic-signs-image/0675.jpg,Có bao nhiêu biển báo giao thông xuất hiện tro...,Có 3 biển báo.,Counting,675,1,3
4,train_000004,0675.jpg,0675.jpg,../traffic-signs-image/0675.jpg,Biển cấm rẽ phải nằm ở vị trí nào?,Nằm treo trên giá long môn phía trên.,Spatial,675,1,4
